In [3]:
import numpy as np 
import pandas as pd  
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings
warnings.filterwarnings("ignore")
print("Libraries Imported")

Libraries Imported


In [4]:
# Load the dataset
data = pd.read_csv(r"D:\SQL Telecom-Customer-Churn-Analysis\WA_Fn-UseC_-Telco-Customer-Churn.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\SQL Telecom-Customer-Churn-Analysis\\WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [ ]:
data


In [ ]:
df = data.copy()

In [ ]:
# View First Rows 
df.head(50) # TO get an overview of the dataset 

In [ ]:
# Check info and Data Types 
df.info() 
#  It tells about
# Count of rows and columns 
# data Type and non null values 

In [ ]:
# Check missing values 
df.isnull().sum()
# There are no missing values 

In [ ]:
# Basic Description
print("Numeric Columns")
print(df.describe())
# It tells about quick overview of summary statistics 
# Count - Number of rows and columns 
# Mean - Average Value 
# Std - How spread out the data in the dataset low std  dev means closely packed or tightly clustered mean the data points are closer to mean 
# high std dev means the data points are further away from the mean more spread out it has more diversity 
# Min - Minimum Number 
# Max - Maximum Number  
# 25% , 50%, 75% Interquartile range  
print()
print()
print()
print()
print("Categoric + Numeric Columns ")
df.describe(include = "all")

In [ ]:
# Check on Duplicate values 
df.duplicated().sum()

In [ ]:
# To see the duplicate rows 
df[df.duplicated()]

In [ ]:
# If duplicate  exists and you want to remove 
df = df.drop_duplicates()

# Categorize columns into numeric and categorical 
# (Very Important )
* Numeric --> Used for scaling 
* Categorical --> Used for Encoding 
* Ml model needs clean seperation 

In [ ]:
""" This dataset contains a mix of:

Categorical columns:

gender, SeniorCitizen, Partner, Dependents,
PhoneService, MultipleLines,
InternetService, OnlineSecurity,
OnlineBackup, DeviceProtection,
TechSupport, StreamingTV, StreamingMovies,
Contract, PaperlessBilling, PaymentMethod,
Churn

Numeric columns:

tenure, MonthlyCharges, TotalCharges

BUT TotalCharges is “object” type even though it contains numbers.

We must convert it. """

In [ ]:
df.dtypes

In [ ]:
# Here Total charges is numeric but stored as string so we are converting it to numeric 
# COnvert to numeric 
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors = "coerce")

In [ ]:
df["TotalCharges"].nunique()

In [ ]:
# Check missing values created during conversion 
df["TotalCharges"].isnull().sum()
# In this we have 11 values like empty  string(" ") so we are going to clean it or convert it  

In [ ]:
# Drop these rows (Required for modelling )
df = df.dropna(subset = ["TotalCharges"])
df

In [ ]:
df.info()

In [ ]:
df.info()

# Now the Dataset is 
* Clean
* No Missing Values 
* Numeric Columnns Fixed 
* Only valid rows kept 
* Ready for Eda and Modeling 

# EDA 

## Churn Distribution 

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts()/len(df)*100

In [ ]:
# Countplot
plt.figure(figsize = (12,8))
sns.countplot(x = "Churn", data = df)
plt.title("Churn Distribution ")
plt.show()

## Here the Dataset is imbalanced (usually ~26% churn)
* We may handle imbalance during modeling (class weight or smote ) to avoid biased predictions 
## There are 5000 plus customers are going  to stay in Telecom Services(not churn)  and 1800 plus  customers are high risk of leaving this service (churn) 
* Here majority stays still predicting this group is more valuable than ones who stys
* The dataset shows 73% of customers has not churned and approximately 27% has churned 

In [ ]:
# Tenure patterns 
plt.figure(figsize = (10,8))
sns.boxplot(x = "Churn",y = "tenure",data =df)
plt.title("Tenure vs Churn")
plt.show()

* Customers with low tenure are likely to high risk of  churning  and  
* Customers with high tenure are loyal to service 
* But we have outliers in Yes column (rare conditions )



# Monthly charge differneces 

In [ ]:
plt.figure(figsize  = (12,8))
sns.boxplot(x = "Churn",y = "MonthlyCharges", data=df)
plt.title("Monthly Charges vs Churn")
plt.show()

* The average of No column is between 60 - 70 
* The average of Yes column  is between 70 - 80 
* Many people have retained when they have low Monthly Charges (between 20 to 80)
* The people who have churned havd high Monthly Charges (between 50 to 100)
# So the Monthly Charges plays a crucial role in Churn

# Contract Type Effect 

In [ ]:
plt.figure(figsize = (12,8))
sns.countplot(x = "Contract", hue = "Churn", data = df) # hue uses for see the second layer of visualization 
plt.xticks(rotation = 45) # To rotate the xlabel or xaxis to 45degree so they dont overlap
plt.title("Contract Type vs Churn")
plt.show()


* In this visualization, people with month to month contracts have high churn rates 
* 1 year or 2 year contracts has been  staying long 


In [ ]:
# Payment Effect 
plt.figure(figsize = (10,8))
sns.countplot(x = "PaymentMethod", hue = "Churn",data = df)

plt.title("Payment Method vs Churn")
plt.show()


* In this case Electronic check customers are churn 
* Mailed check is average or low 
* Both bank transfer and credit card is automatic payment type so it goes fine 
# So electronic check we want to just go see what is the problem like there whether
* The paymnet method is properly functioning  or not 
* The cost of electronic checks were high or not when compared to other options 

In [ ]:
# Internet Effect 
plt.figure(figsize = (12,8))
sns.countplot(x = "InternetService", hue ="Churn",data = df)
plt.title("Internet Service vs Churn")
plt.show()


""" In this visual, Fiber optic Internet Service has high churn when compared to DSL and 
No Internet Service """

# So the problem is with  Fiber optic Internet Service
* Expensive 
* Competitors provide better plan 
* Interruption in signal or technical issue  

# Correlation heatmap

In [ ]:
corr = df[["tenure","MonthlyCharges","TotalCharges"]].corr()
corr

In [ ]:
plt.figure(figsize = (12,8))
sns.heatmap(corr,annot = True,cmap = "coolwarm",linewidth = 0.5)
plt.title("Correlarion Heatmap")
plt.show()

* The tenure and TotalCharges shows a positive correlation of 0.83
* The MonthlyCharges and TotalCharges shows a postive correlation of 0.65
* The tenure and MonthlyCharges shows a weak correlation 
#### By this we came to know Tenure and MonthlyCharges plays a significant role in high risk of churn

# Statistics 

In [ ]:
import scipy.stats as stats 


In [ ]:
df.describe()

In [ ]:
# CHI-SQUARE TEST 
#It is used to see the significant  relationship  between two categories or categoric variables  in the data 
categoric_cols = [
    "gender", "Partner","Dependents","PhoneService",
    "InternetService","OnlineSecurity","OnlineBackup","DeviceProtection",
    "TechSupport","StreamingTV","StreamingMovies","Contract",
    "PaperlessBilling","PaymentMethod"
]

In [ ]:
for col in categoric_cols:
    table = pd.crosstab(df[col],df["Churn"]) # Create a frequency table or contingency table 
    chi2,p,dof,expected = stats.chi2_contingency(table)
    print(f"{col}:p-value = {p}")
    

* Gender and PhoneService shows no singnificant influence with churn statistically
* All other columns have strong relationship with churn


# T Test : It checks whether the average value of a numeric variable  

In [ ]:
Churned_Customers = df[df["Churn"] == "Yes"]["MonthlyCharges"]
NonChurned_Customers = df[df["Churn"] == "No"]["MonthlyCharges"]
stats.ttest_ind(Churned_Customers,NonChurned_Customers,equal_var = False)

* Here p value is 2.65 x 10**-(72) or (2.6573571445160277e-72)
* t-static is 18.34
* df = 4139.66
* p values is small 

In [ ]:
numeric_cols= ["tenure","MonthlyCharges","TotalCharges"]
for col in numeric_cols:
    Customer_Churn = df[df["Churn"] == "Yes"][col]
    Customer_NotChurn = df[df["Churn"] == "No"][col]
    t_stat,p_val = stats.ttest_ind(Customer_Churn,Customer_NotChurn,equal_var = False)
    print(f"{col}:t_stat = {t_stat}, p_value = {p_val}")

# Independent t tests were conducted on all numeric variables  
* By p value we can see that p<0.0001,so there is a significant differences between the two groups 

# Feature Engineering 

In [ ]:
# Drop CustomerID (It is useless in prediction )
df=df.drop("customerID",axis=1)

# We need all data  to be numeric for  ML
### One Hot Encoding 

In [ ]:
df_encoded = pd.get_dummies(df,drop_first = True)
# [ df_encoded = pd.get_dummies(df,drop_first = True, dtype = int64) 
# It will result is 0 and 1 value instead of True and False ]
df_encoded.head()

In [ ]:
#  pd.get_dummies results in Boolean Values by default So we convert it by 0 and 1 using 
df_encoded = df_encoded * 1

In [ ]:
# Create Features (X) and Target (Y)
X = df_encoded.drop("Churn_Yes",axis = 1)
y = df_encoded["Churn_Yes"]

In [ ]:
# Train-Test Split
# We split data into 80% training and 20% testing
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size = 0.2,random_state = 42,stratify = y
)

In [ ]:
# To handle the class imbalance [class_weights ="balanced"] ( pre-defined parameter)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score , classification_report, confusion_matrix)

In [ ]:
log_reg = LogisticRegression(
    max_iter = 1000,
    class_weight = "balanced",
    solver = "liblinear",
    random_state = 42

)
log_reg.fit(X_train,y_train)
y_pred = log_reg.predict(X_test)
print("Accuracy:",accuracy_score(y_test,y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test,y_pred))
print("\nClassification Report:\n",classification_report(y_test,y_pred))

## By Logistic Regression the accuracy is 73%.
### So For class 0,
       * The Precision is 0.91 and Recall is 0.70
#### By this we conclude that Model is very good at predicting customers who will not churn 
### So For class 1,
       * The Precision is 0.49 and Recall is 0.80
#### By this we conclude that Model struggles with predicting churn
###But still the model performs well 
###### For bette accuracy we try another model

In [ ]:
# Import Random Forest 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# The dataset is imbalanced(26%churn,74%non-churn) we use class_weight = "balanced"
rf = RandomForestClassifier(
    n_estimators = 200,
    max_depth = None,
    class_weight = "balanced",
    random_state = 42
)
rf.fit(X_train,y_train)
y_pred_rf = rf.predict(X_test)

In [ ]:
# Evaluate the model
print("Accuracy:",accuracy_score(y_test,y_pred_rf))
print("\nConfusion Matrix\n",confusion_matrix(y_test,y_pred_rf))
print("\nClassification Report\n", classification_report(y_test,y_pred_rf))

In [ ]:
# Most important Fetures 
import pandas as pd 
feat_imp = pd.DataFrame({
    "Feature":X.columns,
    "Importance":rf.feature_importances_
}).sort_values(by = "Importance",ascending = False)
feat_imp.head(15)

In [ ]:
# Import GridSeachCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Defining Parameter Grid 
param_grid = {
    "n_estimators":[100,200,300],
    "max_depth":["None",5,10,20],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4],
    "class_weight":["balanced"]
}

In [ ]:
# Step 3 : Run GridSearchCV
rf = RandomForestClassifier(random_state=42)
grid = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    cv=3,
    scoring = "f1", # optimize for churn F1-score
    n_jobs=-1   # use all CPU scores 
)
grid.fit(X_train,y_train)
print("Best Parameters:",grid.best_params_)

In [ ]:
# Train Final Model with Best Hyperparameters
best_rf = grid.best_estimator_
best_rf.fit(X_train,y_train)
y_pred_best = best_rf.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
print("Accuracy:",accuracy_score(y_test,y_pred_best))
print("\nConfusion Matrix:\n",confusion_matrix(y_test,y_pred_best))
print("\nClassififcation Report:\n", classification_report(y_test,y_pred_best))


In [ ]:
pip install xgboost

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(
    n_estimators = 300,
    learning_rate=0.05,
    max_depth = 6,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = (len(y_train[y_train==0])/len(y_train[y_train==1])),
    eval_metric = "logloss",
    random_state = 42
)
xgb.fit(X_train,y_train)

In [ ]:
y_pred_xgb = xgb.predict(X_test)

In [ ]:
print ("Accuracy:",accuracy_score(y_test,y_pred_xgb))
print("\nConfusion Matrix:\n",confusion_matrix(y_test,y_pred_xgb))
print("\nClassification Report:\n",classification_report(y_test,y_pred_xgb))

In [ ]:
import pandas as pd 
feat_imp = pd.DataFrame({
    "Feature":X.columns,
    "Importance":xgb.feature_importances_
}).sort_values(by = "Importance",ascending = False)
feat_imp.head(15)

# Logistic Regression
# Random Forest
# XGBoost


## Three models were built in this analysis.
### Actually all three were performed well.
### Still Random Forest(tuned) showing some good results when compared to Logistic  Regression and XGBoost
### XGboost is better than Random Forest because, in our scenario 
            *The  dataset is small 
            *Imbalance is handles better by RF class weights 
            *Random Forest is less sensitive to hyperparamaters.